# Bulk RNA-seq TCGA / GEO Public Data Mining Template

本 notebook 用于从 TCGA 下载 bulk RNA-seq count 数据，整理 counts/TPM、临床信息，并完成 Tumor vs Normal 差异分析、单基因表达可视化和生存分析。也可用于读取本地已下载的 GEO / TCGA 表达矩阵。

## 1. Parameter Configuration

In [ ]:
# ===================== Parameter Configuration =====================
# Data source mode
DOWNLOAD_FROM_GDC <- TRUE                 # TRUE = download via TCGAbiolinks; FALSE = load local files
DOWNLOAD_FROM_GEO <- FALSE               # TRUE = download a GEO SeriesMatrix
GEO_ACCESSION <- "GSE12345"             # e.g. "GSE15459"; used only when DOWNLOAD_FROM_GEO = TRUE

# TCGA download parameters
TCGA_PROJECT <- "TCGA-STAD"             # e.g. TCGA-COAD, TCGA-BRCA
TCGA_DATA_CATEGORY <- "Transcriptome Profiling"
TCGA_DATA_TYPE <- "Gene Expression Quantification"
TCGA_WORKFLOW <- "STAR - Counts"
GDC_COUNTS_ASSAY <- NULL                # NULL = auto-detect; otherwise assay name in SE (e.g. "unstranded")
GDC_TPM_ASSAY <- NULL                   # NULL = auto-detect; otherwise assay name in SE (e.g. "tpm_unstrand")

# Local file mode (used when DOWNLOAD_FROM_GDC = FALSE and DOWNLOAD_FROM_GEO = FALSE)
LOCAL_COUNTS_FILE <- "./0-Data/counts.csv"   # genes x samples; first column = gene symbol or row names
LOCAL_TPM_FILE <- "./0-Data/tpm.csv"        # optional; if NULL, TPM is not used
LOCAL_CLINICAL_FILE <- "./0-Data/clinical.csv"
LOCAL_GENE_COLUMN <- NULL                     # set explicitly for numeric Entrez IDs

# Gene symbol mapping (only needed for TCGA download, where row names are ENSEMBL IDs)
GENE_ID_MAP_FILE <- NULL                  # NULL = derive from SummarizedExperiment rowData
                                           # Otherwise: path to file with columns gene_id, gene_name[, gene_type]

# Analysis parameters
MIN_COUNT_PER_SAMPLE_FRAC <- 0.75        # keep genes with count > 1 in >= 75% samples
MIN_COUNT <- 1
TUMOR_NORMAL_DESIGN <- TRUE              # run DESeq2 Tumor vs Normal
DEG_LFC_CUTOFF <- 0.5
DEG_PADJ_CUTOFF <- 0.05

# Survival analysis
GENES_FOR_SURVIVAL <- c("ICAM1")        # genes or signature column names to test by median split
CLINICAL_VARS_FOR_KM <- c("ajcc_pathologic_stage")  # clinical variables for KM; numeric vars are median-split
TIME_UNIT <- "month"                     # "day", "month", or "year"

# Output
OUTDIR <- "RNAseq_TCGA_GEO_Output"
dir.create(OUTDIR, showWarnings = FALSE, recursive = TRUE)

## 2. Environment

In [ ]:
options(stringsAsFactors = FALSE)
# First run if needed:
# install.packages(c("tidyverse", "data.table", "stringr"))
# if (!require("BiocManager", quietly = TRUE)) install.packages("BiocManager")
# BiocManager::install(c("TCGAbiolinks", "SummarizedExperiment", "DESeq2", "clusterProfiler", "org.Hs.eg.db", "survival", "survminer", "GEOquery", "Biobase"))

suppressPackageStartupMessages({
  library(tidyverse)
  library(data.table)
  library(stringr)
  library(SummarizedExperiment)
  library(DESeq2)
  library(clusterProfiler)
  library(org.Hs.eg.db)
  library(survival)
  library(survminer)
})

LIB_DIR <- if (dir.exists("RNAseq_lib")) "RNAseq_lib" else "../RNAseq_lib"
source(file.path(LIB_DIR, "plot_utils.R"))
source(file.path(LIB_DIR, "deg_utils.R"))
source(file.path(LIB_DIR, "enrichment_utils.R"))
source(file.path(LIB_DIR, "tcga_utils.R"))
source(file.path(LIB_DIR, "survival_utils.R"))
source(file.path(LIB_DIR, "geo_utils.R"))
source(file.path(LIB_DIR, "data_utils.R"))
theme_set(theme_publication())
cat("RNAseq_lib:", LIB_DIR, "\n")

## 3. Data Acquisition

In [ ]:
if (DOWNLOAD_FROM_GDC) {
  query <- build_tcga_query(
    project = TCGA_PROJECT,
    data.category = TCGA_DATA_CATEGORY,
    data.type = TCGA_DATA_TYPE,
    workflow.type = TCGA_WORKFLOW
  )
  TCGAbiolinks::GDCdownload(query, directory = file.path(OUTDIR, "GDCdata"))
  se <- TCGAbiolinks::GDCprepare(query, save = TRUE,
                                  save.filename = file.path(OUTDIR, paste0(TCGA_PROJECT, "_mRNA.Rdata")),
                                  directory = file.path(OUTDIR, "GDCdata"))
  assays_list <- extract_tcga_assays(se)
  cat("Available assays:", paste(names(assays_list), collapse = ", "), "\n")

  # Discover count and TPM assays by name instead of hard-coded index
  assay_names <- names(assays_list)
  counts_assay <- if (!is.null(GDC_COUNTS_ASSAY)) GDC_COUNTS_ASSAY else {
    candidates <- assay_names[grepl("count|unstrand", assay_names, ignore.case = TRUE)]
    if (length(candidates) == 0) stop("No count assay found in SummarizedExperiment. Available: ", paste(assay_names, collapse = ", "))
    candidates[1]
  }
  tpm_assay <- if (!is.null(GDC_TPM_ASSAY)) GDC_TPM_ASSAY else {
    candidates <- assay_names[grepl("tpm", assay_names, ignore.case = TRUE)]
    if (length(candidates) == 0) stop("No TPM assay found in SummarizedExperiment. Available: ", paste(assay_names, collapse = ", "))
    candidates[1]
  }
  if (!counts_assay %in% assay_names) stop("Requested count assay not found: ", counts_assay)
  if (!tpm_assay %in% assay_names) stop("Requested TPM assay not found: ", tpm_assay)
  cat("Using count assay:", counts_assay, "; TPM assay:", tpm_assay, "\n")

  # Map ENSEMBL IDs to gene symbols and de-duplicate
  if (is.null(GENE_ID_MAP_FILE)) {
    id_map <- build_id_map_from_se(se, id_col = "gene_id", symbol_col = "gene_name", type_col = "gene_type")
    cat("Derived gene ID map from SummarizedExperiment rowData:", nrow(id_map), "rows\n")
  } else {
    id_map <- read.delim(GENE_ID_MAP_FILE, stringsAsFactors = FALSE)
  }
  counts_raw <- symbolize_and_dedup(assays_list[[counts_assay]], id_map, id_col = "gene_id", symbol_col = "gene_name")
  tpm_raw <- symbolize_and_dedup(assays_list[[tpm_assay]], id_map, id_col = "gene_id", symbol_col = "gene_name")
  clinical_raw <- extract_tcga_clinical(se)
} else if (DOWNLOAD_FROM_GEO) {
  gse <- download_geo_series_matrix(GEO_ACCESSION, destdir = file.path(OUTDIR, "0-Data"))
  geo <- parse_geo_series_matrix(gse)
  counts_raw <- prepare_geo_counts(geo$expr, geo$feature)
  clinical_raw <- geo$pdata
  tpm_raw <- NULL
  cat("GEO assay:", nrow(counts_raw), "genes x", ncol(counts_raw), "samples\n")
} else {
  counts_raw <- read_expression_matrix(LOCAL_COUNTS_FILE, gene_column = LOCAL_GENE_COLUMN)
  tpm_raw <- if (!is.null(LOCAL_TPM_FILE) && file.exists(LOCAL_TPM_FILE)) {
    read_expression_matrix(LOCAL_TPM_FILE, gene_column = LOCAL_GENE_COLUMN)
  } else NULL
  if (!is.null(tpm_raw)) {
    validate_expression_contract(tpm_raw, expected = "tpm")
    validate_samples_match(colnames(tpm_raw), colnames(counts_raw), context = "TPM vs counts")
  }
  clinical_raw <- read_metadata(LOCAL_CLINICAL_FILE, sample_column = "barcode", required_columns = "barcode")
}

# DESeq2 is valid only for raw integer counts. GEO SeriesMatrix values are often
# normalized microarray/log expression and will fail here; use limma for those data.
tryCatch(validate_count_matrix(counts_raw), error = function(e) {
  stop("Differential-expression input is not raw integer counts: ", conditionMessage(e),
       " For normalized GEO SeriesMatrix data, use a limma workflow or obtain raw RNA-seq counts.")
})
if (!is.null(tpm_raw)) validate_expression_contract(tpm_raw, expected = "tpm")
cat("Counts:", nrow(counts_raw), "genes x", ncol(counts_raw), "samples\n")
if (!is.null(tpm_raw)) cat("TPM:", nrow(tpm_raw), "genes x", ncol(tpm_raw), "samples\n")

## 4. Clinical Cleanup and Sample Filtering

In [ ]:
# Standardize column names if needed
colnames(clinical_raw) <- make.names(colnames(clinical_raw), unique = TRUE)

# Infer tumor/normal grouping
if ("barcode" %in% colnames(clinical_raw)) {
  clinical_raw$tissue_type <- infer_tcga_tumor_normal(clinical_raw$barcode)
}

# Validate sample overlap
validate_samples_match(colnames(counts_raw), clinical_raw$barcode, context = "counts vs clinical")

# Keep only samples present in counts
common_samples <- intersect(colnames(counts_raw), clinical_raw$barcode)
counts_raw <- counts_raw[, common_samples, drop = FALSE]
if (!is.null(tpm_raw)) tpm_raw <- tpm_raw[, common_samples, drop = FALSE]
clinical <- clinical_raw[match(common_samples, clinical_raw$barcode), ]
clinical$condition <- factor(clinical$tissue_type, levels = c("Normal", "Tumor"))
if (TUMOR_NORMAL_DESIGN && (anyNA(clinical$condition) || !all(c("Normal", "Tumor") %in% clinical$condition))) {
  stop("Tumor/Normal design requires every retained sample to have a valid tissue type and both groups to be present.")
}

write.csv(clinical, file.path(OUTDIR, "clinical_clean.csv"), row.names = FALSE)
print(table(clinical$condition, useNA = "ifany"))

## 5. DESeq2 Tumor vs Normal DEG

In [ ]:
if (TUMOR_NORMAL_DESIGN) {
  keep_genes <- rowSums(counts_raw > MIN_COUNT) >= ceiling(MIN_COUNT_PER_SAMPLE_FRAC * ncol(counts_raw))
  counts_filt <- counts_raw[keep_genes, ]
  counts_filt <- as.matrix(counts_filt)
  mode(counts_filt) <- "numeric"

  colData <- data.frame(row.names = colnames(counts_filt), condition = clinical$condition)
  dds <- DESeqDataSetFromMatrix(countData = counts_filt, colData = colData, design = ~ condition)
  dds <- DESeq(dds)
  res <- results(dds, contrast = c("condition", "Tumor", "Normal"))
  res_df <- as.data.frame(res) |>
    rownames_to_column("gene_name") |>
    arrange(padj)
  write.csv(res_df, file.path(OUTDIR, "DESeq2_Tumor_vs_Normal.csv"), row.names = FALSE)

  sig_genes <- res_df |>
    filter(!is.na(padj), padj < DEG_PADJ_CUTOFF, abs(log2FoldChange) > DEG_LFC_CUTOFF) |>
    pull(gene_name)
  cat("Significant DEGs:", length(sig_genes), "
")
}


## 6. DEG Visualization

In [ ]:
if (TUMOR_NORMAL_DESIGN) {
  plot_volcano_pdf(
    res_df, comp_name = "Tumor_vs_Normal",
    pvalue_thresh = DEG_PADJ_CUTOFF, log2fc_thresh = DEG_LFC_CUTOFF,
    filename = file.path(OUTDIR, "Volcano_Tumor_vs_Normal.pdf"),
    pvalue_column = "padj", lfc_column = "log2FoldChange"
  )
}


## 7. Single-Gene Expression and Survival

In [ ]:
if (!is.null(tpm_raw)) {
  expr_log <- log2(as.matrix(tpm_raw) + 1)
} else {
  # Never use log2(raw counts + 1) for sample comparisons because library size
  # remains a confounder. Derive VST expression from the validated raw counts.
  dds_for_expression <- if (exists("dds")) dds else {
    DESeqDataSetFromMatrix(as.matrix(counts_raw),
      colData = data.frame(row.names = colnames(counts_raw)), design = ~ 1)
  }
  expr_log <- assay(vst(dds_for_expression, blind = !TUMOR_NORMAL_DESIGN))
}

for (gene in GENES_FOR_SURVIVAL) {
  if (!gene %in% rownames(expr_log)) {
    warning("Gene not found: ", gene)
    next
  }
  # Boxplot by condition
  plot_tcga_gene_boxplot_pdf(
    expr_log, gene, clinical$condition,
    filename = file.path(OUTDIR, paste0("Expression_boxplot_", gene, ".pdf")),
    title = paste(gene, "expression")
  )

  # Survival (tumor samples only)
  surv_df <- prepare_tcga_survival(clinical)
  expr_vec <- expr_log[gene, surv_df$barcode]
  surv_df[[gene]] <- as.numeric(expr_vec)
  plot_km_by_median_pdf(
    surv_df, value_col = gene,
    filename = file.path(OUTDIR, paste0("KM_", gene, ".pdf")),
    title = paste(gene, "survival"),
    time_unit = TIME_UNIT
  )
}


## 7.5 Survival Analysis (KM + Univariate/Multivariate Cox)

In [ ]:
# Prepare survival data
surv_df <- prepare_tcga_survival(clinical)
cat("Survival samples:", nrow(surv_df), "
")
cat("Events:", sum(surv_df$status), "
")

# KM by median expression for each gene
for (gene in GENES_FOR_SURVIVAL) {
  if (!gene %in% rownames(expr_log)) next
  surv_df[[gene]] <- as.numeric(expr_log[gene, surv_df$barcode])

  # Median-split KM
  plot_km_by_median_pdf(
    surv_df, value_col = gene,
    filename = file.path(OUTDIR, paste0("KM_", gene, ".pdf")),
    title = paste(gene, "survival"),
    time_unit = TIME_UNIT
  )

  # Quartile-split KM
  surv_df[[paste0(gene, "_quartile")]] <- stratify_by_quantile(surv_df[[gene]], n_groups = 4)
  plot_km_by_group_pdf(
    surv_df, group_col = paste0(gene, "_quartile"),
    filename = file.path(OUTDIR, paste0("KM_quartile_", gene, ".pdf")),
    title = paste(gene, "quartile survival"),
    time_unit = TIME_UNIT
  )
}

# KM by clinical variables (categorical or median-split numeric)
if (length(CLINICAL_VARS_FOR_KM) > 0) {
  run_clinical_km(
    surv_df,
    clinical_df = clinical,
    var_cols = CLINICAL_VARS_FOR_KM,
    outdir = OUTDIR,
    time_unit = TIME_UNIT
  )
}

# Univariate Cox for all genes of interest
surv_vars <- intersect(GENES_FOR_SURVIVAL, colnames(surv_df))
if (length(surv_vars) > 0) {
  uni_cox <- run_univariate_cox(surv_df, vars = surv_vars)
  if (!is.null(uni_cox)) {
    write.csv(uni_cox, file.path(OUTDIR, "univariate_Cox.csv"), row.names = FALSE)
    plot_cox_forest_pdf(
      uni_cox,
      filename = file.path(OUTDIR, "univariate_Cox_forest.pdf"),
      title = "Univariate Cox Regression"
    )
  }
}

# Multivariate Cox using all genes of interest
if (length(surv_vars) >= 2) {
  multi_cox <- run_multivariate_cox(surv_df, vars = surv_vars)
  if (!is.null(multi_cox)) {
    write.csv(multi_cox, file.path(OUTDIR, "multivariate_Cox.csv"), row.names = FALSE)
    plot_cox_forest_pdf(
      multi_cox,
      filename = file.path(OUTDIR, "multivariate_Cox_forest.pdf"),
      title = "Multivariate Cox Regression"
    )
  }
}


## 8. ORA and GSEA


In [ ]:
if (TUMOR_NORMAL_DESIGN) {
  # ORA
  deg_list <- genes_for_enrichment(
    res_df, pvalue_thresh = DEG_PADJ_CUTOFF, log2fc_thresh = DEG_LFC_CUTOFF,
    pvalue_column = "padj", lfc_column = "log2FoldChange"
  )
  universe <- map_symbols_to_entrez(rownames(counts_raw), org.Hs.eg.db)
  ego <- run_go_ora(deg_list$sig, org_db = org.Hs.eg.db, universe = universe$ENTREZID)
  ekegg <- run_kegg_ora(deg_list$sig, org_db = org.Hs.eg.db, universe = universe$ENTREZID, organism = "hsa")
  if (!is.null(ego)) {
    write.csv(as.data.frame(ego), file.path(OUTDIR, "GO_ORA_Tumor_vs_Normal.csv"), row.names = FALSE)
    plot_enrich_suite_pdf(ego, file.path(OUTDIR, "GO_ORA_Tumor_vs_Normal"), "GO ORA")
  }
  if (!is.null(ekegg)) {
    write.csv(as.data.frame(ekegg), file.path(OUTDIR, "KEGG_ORA_Tumor_vs_Normal.csv"), row.names = FALSE)
    plot_enrich_suite_pdf(ekegg, file.path(OUTDIR, "KEGG_ORA_Tumor_vs_Normal"), "KEGG ORA")
  }

  # GSEA
  ranked <- ranked_gene_list(res_df, rank_column = "stat")
  entrez_ranked <- make_entrez_ranked_list(ranked, org.Hs.eg.db)
  gsea_go <- run_go_gsea(entrez_ranked, org_db = org.Hs.eg.db)
  gsea_kegg <- run_kegg_gsea(entrez_ranked, organism = "hsa")
  if (!is.null(gsea_go)) {
    write_gsea_tables(gsea_go, file.path(OUTDIR, "GO_GSEA_Tumor_vs_Normal.csv"))
    plot_gsea_suite_pdf(gsea_go, file.path(OUTDIR, "GO_GSEA_Tumor_vs_Normal"), "GO GSEA")
  }
  if (!is.null(gsea_kegg)) {
    write_gsea_tables(gsea_kegg, file.path(OUTDIR, "KEGG_GSEA_Tumor_vs_Normal.csv"))
    plot_gsea_suite_pdf(gsea_kegg, file.path(OUTDIR, "KEGG_GSEA_Tumor_vs_Normal"), "KEGG GSEA")
  }
}


### 8.1 Publication-Grade Theme Dot-heatmap

Group ORA/GSEA terms into biological themes for a manuscript-ready overview.

In [ ]:
if (TUMOR_NORMAL_DESIGN) {
  theme_outdir <- file.path(OUTDIR, "ThemeEnrichment")
  dir.create(theme_outdir, showWarnings = FALSE, recursive = TRUE)

  go_ora_map <- list()
  kegg_ora_map <- list()
  go_gsea_map <- list()
  kegg_gsea_map <- list()
  if (exists("ego") && !is.null(ego)) go_ora_map[["Tumor_vs_Normal"]] <- ego
  if (exists("ekegg") && !is.null(ekegg)) kegg_ora_map[["Tumor_vs_Normal"]] <- ekegg
  if (exists("gsea_go") && !is.null(gsea_go)) go_gsea_map[["Tumor_vs_Normal"]] <- gsea_go
  if (exists("gsea_kegg") && !is.null(gsea_kegg)) kegg_gsea_map[["Tumor_vs_Normal"]] <- gsea_kegg

  theme_defs <- default_enrichment_themes()

  if (length(go_ora_map) > 0) {
    p <- plot_theme_dotheatmap_from_results(go_ora_map, file.path(theme_outdir, "Theme_dotheatmap_GO_ORA.pdf"),
      title = "GO ORA Biological Themes", subtitle = "GO-BP ORA | Tumor vs Normal", theme_defs = theme_defs, ontology_filter = "BP")
    if (!is.null(p)) print(p)
  }
  if (length(kegg_ora_map) > 0) {
    p <- plot_theme_dotheatmap_from_results(kegg_ora_map, file.path(theme_outdir, "Theme_dotheatmap_KEGG_ORA.pdf"),
      title = "KEGG ORA Pathway Themes", subtitle = "KEGG ORA | Tumor vs Normal", theme_defs = theme_defs, ontology_filter = NULL)
    if (!is.null(p)) print(p)
  }
  if (length(go_gsea_map) > 0) {
    p <- plot_theme_dotheatmap_from_results(go_gsea_map, file.path(theme_outdir, "Theme_dotheatmap_GO_GSEA.pdf"),
      title = "GO GSEA Biological Themes", subtitle = "GO-BP GSEA | Tumor vs Normal", theme_defs = theme_defs, ontology_filter = "BP")
    if (!is.null(p)) print(p)
  }
  if (length(kegg_gsea_map) > 0) {
    p <- plot_theme_dotheatmap_from_results(kegg_gsea_map, file.path(theme_outdir, "Theme_dotheatmap_KEGG_GSEA.pdf"),
      title = "KEGG GSEA Pathway Themes", subtitle = "KEGG GSEA | Tumor vs Normal", theme_defs = theme_defs, ontology_filter = NULL)
    if (!is.null(p)) print(p)
  }
}


### 8.2 Publication-Grade Single-Term GSEA Figures

Generate one running-enrichment figure per selected GSEA term.

In [ ]:
if (TUMOR_NORMAL_DESIGN) {
  single_term_outdir <- file.path(theme_outdir, "single_term_gsea")
  dir.create(single_term_outdir, showWarnings = FALSE, recursive = TRUE)

  if (exists("gsea_go") && !is.null(gsea_go) && nrow(as.data.frame(gsea_go)) > 0) {
    ggo_df <- significant_gsea_terms(gsea_go)
    ggo_df <- ggo_df[order(ggo_df$p.adjust, -abs(ggo_df$NES)), ]
    top_terms <- rbind(utils::head(ggo_df[ggo_df$NES > 0, ], 3),
                       utils::head(ggo_df[ggo_df$NES < 0, ], 3))
    plot_gsea_term_figures_from_df(gsea_go, top_terms,
      outdir = file.path(single_term_outdir, "GO_Tumor_vs_Normal"),
      contrast_label = "Tumor vs Normal", prefix = "gseaplot2_GO")
  }
  if (exists("gsea_kegg") && !is.null(gsea_kegg) && nrow(as.data.frame(gsea_kegg)) > 0) {
    gkegg_df <- significant_gsea_terms(gsea_kegg)
    gkegg_df <- gkegg_df[order(gkegg_df$p.adjust, -abs(gkegg_df$NES)), ]
    top_terms <- rbind(utils::head(gkegg_df[gkegg_df$NES > 0, ], 3),
                       utils::head(gkegg_df[gkegg_df$NES < 0, ], 3))
    plot_gsea_term_figures_from_df(gsea_kegg, top_terms,
      outdir = file.path(single_term_outdir, "KEGG_Tumor_vs_Normal"),
      contrast_label = "Tumor vs Normal", prefix = "gseaplot2_KEGG")
  }
}


## 9. Save Session


In [ ]:
save.image(file = file.path(OUTDIR, "TCGA_GEO_workspace.Rdata"))
writeLines(capture.output(sessionInfo()), file.path(OUTDIR, "sessionInfo.txt"))
cat("Analysis complete. Outputs saved to", OUTDIR, "
")
